In [0]:
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.sql.window import Window

events_table = "workspace.ecommerce.ecommerce_delta_table"
gold_table = "workspace.ecommerce.user_recommendations_gold"

In [0]:
events_df = spark.read.table(events_table)

ratings = (
    events_df
    .filter(F.col("event_type").isin("view", "cart", "purchase"))
    .withColumn(
        "rating",
        F.when(F.col("event_type") == "purchase", 3)
         .when(F.col("event_type") == "cart", 2)
         .otherwise(1)
    )
    .groupBy("user_id", "product_id")
    .agg(F.sum("rating").alias("rating"))
)

print("Total interactions:", ratings.count())

In [0]:
als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",
    rank=8,
    maxIter=5,
    regParam=0.1,
    coldStartStrategy="drop"
)

model = als.fit(ratings)
print("ALS model trained")

In [0]:
# Sample users for fast scoring (challenge-friendly)
sample_users = ratings.select("user_id").distinct().sample(0.01, seed=42)

# Small product sample
products = ratings.select("product_id").distinct().sample(0.1, seed=42)

candidates = sample_users.crossJoin(products)

print("Candidate size:", candidates.count())

In [0]:
predictions = model.transform(candidates).filter(F.col("prediction").isNotNull())

In [0]:
window = Window.partitionBy("user_id").orderBy(F.col("prediction").desc())

top5 = (
    predictions
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") <= 5)
    .select("user_id", "product_id", F.col("prediction").alias("score"))
)

top5.show(10)

In [0]:
top5.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_table)

print("Saved:", gold_table)

In [0]:
spark.sql(f"""
SELECT user_id, COUNT(*) as recommendations
FROM {gold_table}
GROUP BY user_id
ORDER BY recommendations DESC
LIMIT 10
""").display()